In [ ]:
#we have 2 problem:
#1: we need the playwright browser to save steam's session first 
#2: have the session infos to be used on later steps, when we login to other site with steams account -> get the session infos -> use it to fetch data.
from playwright.async_api import async_playwright
async with async_playwright() as p:
    browser = await p.chromium.launch(headless=False)  # headless=False opens a visible window
    context = await browser.new_context()
    
    page = await browser.new_page()
    await page.goto("https://buff.163.com",wait_until="domcontentloaded", timeout=120000)

    # Wait for login button or perform manual login
    await page.wait_for_selector("text=Login/Register", timeout=120000)
    await page.click("text=Login/Register")
    # Prepare to catch the popup (Steam login)
    async with page.expect_popup() as popup_info:
        await page.is_visible("text=Other login methods",timeout=120000)
        await page.click("text=Other login methods")
        
    #inputing account's info ( im using an freshly created account, without steam guard and such)
    popup = await popup_info.value

    login_section = popup.locator("div.page_content").locator("div[data-featuretarget='login']")
    username_input = login_section.locator("div", has_text="Sign in with account name").locator("input[type='text']")
    await username_input.is_visible(timeout=30000)
    password_input = login_section.locator("div", has_text="Password").locator("input[type='password']")
    await password_input.is_visible(timeout=30000)

    # Fill credentials
    await username_input.fill("username")
    await password_input.fill("password")
    await login_section.locator("button", has_text="Sign in").click()

    # Click the sign-in button to login buff
    final_login_form = popup.locator("form#openidForm")
    sign_in_button = final_login_form.locator("input[type='submit']", has_text="Sign In")

    # Wait for it to be visible and click
    await sign_in_button.wait_for(state="visible", timeout=30000)
    await sign_in_button.click()


    # Close the browser when done
    browser.close()